In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

## First Look at the Data

Before doing anything else, we check the basic structure of the data:
how many rows and columns we have, what type each column is and how
many values are missing in each column. This helps us understand what
needs to be cleaned before we can use this data for modeling

In [81]:
df = pd.read_csv('data/movies.csv' , encoding='latin-1')
df.head()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  str    
 1   Year      14981 non-null  str    
 2   Duration  7240 non-null   str    
 3   Genre     13632 non-null  str    
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   str    
 6   Director  14984 non-null  str    
 7   Actor 1   13892 non-null  str    
 8   Actor 2   13125 non-null  str    
 9   Actor 3   12365 non-null  str    
dtypes: float64(1), str(9)
memory usage: 1.2 MB


In [83]:
df.shape

(15509, 10)

In [84]:
df.isnull().sum()

Name           0
Year         528
Duration    8269
Genre       1877
Rating      7590
Votes       7589
Director     525
Actor 1     1617
Actor 2     2384
Actor 3     3144
dtype: int64

The year and votes column look like numbers but stored as text where year is sourrounded by brackets and 
votes ara comma separated

In [85]:
df[['Year', 'Votes']].head(10)

,Year,Votes
0,NaN,NaN
1,(2019),8
2,(2021),NaN
3,(2019),35
4,(2010),NaN
5,(1997),827
6,(2005),"1,086"
7,(2008),NaN
8,(2012),326
9,(2014),11


## Cleaing year column 
Removing brackets from year column and converting it to numbers
we use `errors='coerce'`, which turns any value that cannot be converted into a missing value instead of stopping the whole process with an error

In [86]:
df['Year'] =df['Year'].str.replace('(', '', regex=False)
df['Year']= df['Year'].str.replace(')', '', regex=False)
df['Year'] =pd.to_numeric(df['Year'], errors='coerce')
df['Year'].head(10)

0       NaN
1    2019.0
2    2021.0
3    2019.0
4    2010.0
5    1997.0
6    2005.0
7    2008.0
8    2012.0
9    2014.0
Name: Year, dtype: float64

## Cleaning votes column
Removing commas from large numbers and managing errors as same as year

In [87]:
df['Votes']=df['Votes'].str.replace(',', '', regex=False)
df['Votes']= pd.to_numeric(df['Votes'],errors='coerce')
df['Votes'].head(10)

0       NaN
1       8.0
2       NaN
3      35.0
4       NaN
5     827.0
6    1086.0
7       NaN
8     326.0
9      11.0
Name: Votes, dtype: float64

## Checking and cleaning the durarion column
The Duration column has the word "min" attached to every value, like
"109 min". We remove this text and convert the column into a proper
number representing minutes

In [88]:
df['Duration'].head(10)

0        NaN
1    109 min
2     90 min
3    110 min
4    105 min
5    147 min
6    142 min
7     59 min
8     82 min
9    116 min
Name: Duration, dtype: str

In [89]:
df['Duration'] = df['Duration'].str.replace(' min', '', regex=False)
df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce')
df['Duration'].head(10)

0      NaN
1    109.0
2     90.0
3    110.0
4    105.0
5    147.0
6    142.0
7     59.0
8     82.0
9    116.0
Name: Duration, dtype: float64

## Handling Missing Values

Now that our numeric columns are properly cleaned, we need to decide
what to do with the missing values in each column. Different columns
need different treatment depending on how important they are and how
much data is missing.

Our plan:
- Rating: This is what we are trying to predict, so any row missing
  a Rating is useless for training. We will drop these rows entirely.
- Votes: Almost always missing exactly when Rating is missing, so
  these rows get dropped along with Rating anyway.
- Duration: Over half the values are missing. We will fill missing
  values with the median duration, since the median is not affected
  much by extreme outliers.
- Year: Only a small number missing. We will fill these with the
  median year.
- Genre, Director, Actor 1/2/3: These are text columns. We will fill
  missing values with the label "Unknown" instead of guessing a name.

In [90]:
df = df.dropna(subset=['Rating']) #Dropping rating with missing values
df.shape

(7919, 10)

## Checking Votes Column

We already dropped rows where Rating was missing. Votes was missing
in almost the same rows as Rating. Let us check how many missing
values are left in Votes now.

In [91]:
df['Votes'].isnull().sum()

np.int64(0)

## Filling Missing Values

Duration and Year are numbers. We fill missing values with the median.
Median is not affected much by very high or very low values.

Genre Director Actor 1 Actor 2 Actor 3 are text columns. We cannot
average a name or a genre. So we fill missing values with the word
"Unknown".

In [92]:
df['Duration'] = df['Duration'].fillna(df['Duration'].median())
df['Year'] = df['Year'].fillna(df['Year'].median())

df['Genre'] = df['Genre'].fillna('Unknown')
df['Director'] = df['Director'].fillna('Unknown')
df['Actor 1'] = df['Actor 1'].fillna('Unknown')
df['Actor 2'] = df['Actor 2'].fillna('Unknown')
df['Actor 3'] = df['Actor 3'].fillna('Unknown')

df.isnull().sum()

Name        0
Year        0
Duration    0
Genre       0
Rating      0
Votes       0
Director    0
Actor 1     0
Actor 2     0
Actor 3     0
dtype: int64

## Final Check After Cleaning

Let us check the dataset one more time. This confirms every column
has no missing values left, except Name and Year which we already
handled, and that all columns have the correct data type.

In [93]:
df.isnull().sum()

Name        0
Year        0
Duration    0
Genre       0
Rating      0
Votes       0
Director    0
Actor 1     0
Actor 2     0
Actor 3     0
dtype: int64

In [94]:
df.info()

<class 'pandas.DataFrame'>
Index: 7919 entries, 1 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      7919 non-null   str    
 1   Year      7919 non-null   float64
 2   Duration  7919 non-null   float64
 3   Genre     7919 non-null   str    
 4   Rating    7919 non-null   float64
 5   Votes     7919 non-null   float64
 6   Director  7919 non-null   str    
 7   Actor 1   7919 non-null   str    
 8   Actor 2   7919 non-null   str    
 9   Actor 3   7919 non-null   str    
dtypes: float64(4), str(6)
memory usage: 680.5 KB


## Dropping the Name Column

Name does not help predict rating. It is just a label for each movie.
We remove it before building the model. We keep it out of the model
input but the original column stays in the file if needed later.

In [95]:
df = df.drop('Name', axis=1)
df.head()

,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
1,2019.0,109.0,Drama,7.0,8.0,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
3,2019.0,110.0,"Comedy, Romance",4.4,35.0,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
5,1997.0,147.0,"Comedy, Drama, Musical",4.7,827.0,Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,Shammi Kapoor
6,2005.0,142.0,"Drama, Romance, War",7.4,1086.0,Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,Yashpal Sharma
8,2012.0,82.0,"Horror, Mystery, Thriller",5.6,326.0,Allyson Patel,Yash Dave,Muntazir Ahmad,Kiran Bhatia


In [96]:
df.columns

Index(['Year', 'Duration', 'Genre', 'Rating', 'Votes', 'Director', 'Actor 1',
       'Actor 2', 'Actor 3'],
      dtype='str')

## Checking Columns and Unique Values

Confirm the final list of columns after dropping Name then we
check how many unique values each text column has. This decides how
we encode them into numbers.

In [97]:
df.columns

Index(['Year', 'Duration', 'Genre', 'Rating', 'Votes', 'Director', 'Actor 1',
       'Actor 2', 'Actor 3'],
      dtype='str')

In [98]:
print(df['Genre'].nunique())
print(df['Director'].nunique())
print(df['Actor 1'].nunique())
print(df['Actor 2'].nunique())
print(df['Actor 3'].nunique())

433
3140
2552
2874
3065


## Encoding Director and Actor Columns

Director and Actor columns have too many unique values for one hot
encoding. Instead we use frequency encoding. We replace each name
with the number of times it appears in the dataset. This keeps the
number of columns the same and still gives the model useful
information

In [99]:
director_freq = df['Director'].value_counts()
df['Director_Freq'] = df['Director'].map(director_freq)

actor1_freq = df['Actor 1'].value_counts()
df['Actor1_Freq'] = df['Actor 1'].map(actor1_freq)

actor2_freq = df['Actor 2'].value_counts()
df['Actor2_Freq'] = df['Actor 2'].map(actor2_freq)

actor3_freq = df['Actor 3'].value_counts()
df['Actor3_Freq'] = df['Actor 3'].map(actor3_freq)

df[['Director', 'Director_Freq', 'Actor 1', 'Actor1_Freq']].head(10)

,Director,Director_Freq,Actor 1,Actor1_Freq
1,Gaurav Bakshi,1,Rasika Dugal,2
3,Ovais Khan,1,Prateik,5
5,Rahul Rawail,17,Bobby Deol,18
6,Shoojit Sircar,7,Jimmy Sheirgill,25
8,Allyson Patel,1,Yash Dave,1
9,Biju Bhaskar Nair,1,Augustine,1
10,Madhu Ambat,1,Rati Agnihotri,17
11,Arshad Siddiqui,2,Pankaj Berry,5
12,Partho Ghosh,20,Jackie Shroff,55
13,Rabi Kinagi,1,Jeet,2


## Checking Genre Format

Genre can have more than one value in a single cell like Drama Musical
separated by a comma. We look at a few rows before deciding how to
encode it.

In [100]:
df['Genre'].head(10)

1                         Drama
3               Comedy, Romance
5        Comedy, Drama, Musical
6           Drama, Romance, War
8     Horror, Mystery, Thriller
9       Action, Crime, Thriller
10                        Drama
11                       Horror
12    Horror, Romance, Thriller
13       Comedy, Drama, Romance
Name: Genre, dtype: str

## Encoding Genre

A movie can belong to more than one genre. We split the Genre column
on the comma and create one column for each genre. Each new column
has 1 if the movie belongs to that genre and 0 if it does not. This
way we do not lose information about movies with more than one genre.

In [101]:
genre_dummies = df['Genre'].str.get_dummies(', ')
df = pd.concat([df, genre_dummies], axis=1)
df.head()

,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3,Director_Freq,...,Musical,Mystery,News,Romance,Sci-Fi,Sport,Thriller,Unknown,War,Western
1,2019.0,109.0,Drama,7.0,8.0,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid,1,...,0,0,0,0,0,0,0,0,0,0
3,2019.0,110.0,"Comedy, Romance",4.4,35.0,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor,1,...,0,0,0,1,0,0,0,0,0,0
5,1997.0,147.0,"Comedy, Drama, Musical",4.7,827.0,Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,Shammi Kapoor,17,...,1,0,0,0,0,0,0,0,0,0
6,2005.0,142.0,"Drama, Romance, War",7.4,1086.0,Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,Yashpal Sharma,7,...,0,0,0,1,0,0,0,0,1,0
8,2012.0,82.0,"Horror, Mystery, Thriller",5.6,326.0,Allyson Patel,Yash Dave,Muntazir Ahmad,Kiran Bhatia,1,...,0,1,0,0,0,0,1,0,0,0


## Dropping Original Text Columns

We already created number versions of Genre Director and Actor
columns. The original text columns are no longer needed for the
model. We drop them now.

In [102]:
df = df.drop(['Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3'], axis=1)
df.columns

Index(['Year', 'Duration', 'Rating', 'Votes', 'Director_Freq', 'Actor1_Freq',
       'Actor2_Freq', 'Actor3_Freq', 'Action', 'Adventure', 'Animation',
       'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family',
       'Fantasy', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News',
       'Romance', 'Sci-Fi', 'Sport', 'Thriller', 'Unknown', 'War', 'Western'],
      dtype='str')

## Splitting Features and Target

We separate the data into two parts. X holds all the columns we use
to predict. y holds the Rating column, which is what we want to
predict. The model learns the relationship between X and y.

In [103]:
X = df.drop('Rating', axis=1)
y = df['Rating']

X.shape, y.shape

((7919, 30), (7919,))

## Splitting into Train and Test Sets

We split the data into a training set and a testing set. The model
learns patterns only from the training set. We use the testing set
later to check how well the model performs on data it has never seen.
This is important because checking performance on data the model
already learned from would give a false sense of accuracy.

In [104]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape


((6335, 30), (1584, 30))

## Training a Linear Regression Model

We start with Linear Regression as a baseline model. It is simple and
fast. It tries to find a straight line relationship between the
features and the rating. Later we compare it with a more complex
model to see if performance improves.

In [105]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

## Evaluating Linear Regression

We check how close the predicted ratings are to the actual ratings.
We use three common metrics. MAE shows the average error in rating
points. RMSE also shows average error but punishes larger mistakes
more. R2 score shows how much of the rating's pattern the model was
able to explain, from 0 to 1, higher is better.

In [106]:
mae = mean_absolute_error(y_test, lr_predictions)
rmse = mean_squared_error(y_test, lr_predictions) ** 0.5
r2 = r2_score(y_test, lr_predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 0.9745603526833874
RMSE: 1.2277717615732147
R2 Score: 0.1891846066436661


## Trying Random Forest Regression

Linear Regression only explained a small part of the pattern. We now
try Random Forest, a model that can capture more complex non linear
relationships between features and rating. We compare its performance
with Linear Regression.

In [107]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, rf_predictions)
rmse_rf = mean_squared_error(y_test, rf_predictions) ** 0.5
r2_rf = r2_score(y_test, rf_predictions)

print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

MAE: 0.8071414141414139
RMSE: 1.0731045675416668
R2 Score: 0.3806005848038303


## Checking Feature Importance

Random Forest can tell us which features mattered most when making
predictions. This helps us understand what actually drives movie
ratings according to our model.

In [108]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

importances.head(10)

Votes            0.238549
Year             0.213696
Duration         0.099883
Actor1_Freq      0.068086
Director_Freq    0.064051
Actor3_Freq      0.062012
Actor2_Freq      0.060865
Documentary      0.037873
Action           0.021958
Drama            0.020479
dtype: float64

## Conclusion

We built a model to predict movie ratings using year, duration, votes,
genre, director and actor information. After cleaning the data and
encoding text columns into numbers, we trained two models. Random
Forest performed better than Linear Regression, with an R2 score of
0.38 compared to 0.19. Votes and Year were the most important features
in predicting rating. The model still has room for improvement, since
many real factors that affect a movie's rating, like story quality or
audience reception, are not present in this dataset.